In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from io import BytesIO
import base64
from pathlib import Path

In [ ]:
def fig_to_b64(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=150)
    return base64.b64encode(buf.getvalue()).decode()

def img_tag(b64): return f'<img src="data:image/png;base64,{b64}" style="width:100%;margin:1em 0">'

In [ ]:
def portfolio_report(*portfolios, path=None):
    def plot_html(ax):
        fig = ax.get_figure()
        html = img_tag(fig_to_b64(fig))
        plt.close(fig)
        return html

    single = len(portfolios) == 1
    sections = []

    # --- Portfolio Stats ---
    sections.append("<h2>Portfolio Stats</h2>")
    sections.append("<h3>Summary</h3>")
    sections.append(compare(*portfolios).to_html())

    if single:
        sections.append("<h3>Asset Stats</h3>")
        sections.append(portfolios[0].asset_stats().to_html())
        sections.append("<h3>Correlation Matrix</h3>")
        sections.append(portfolios[0].correlation().to_html())

    # --- Portfolio Returns ---
    sections.append("<h2>Portfolio Returns</h2>")
    sections.append("<h3>Cumulative Excess Return</h3>")
    sections.append(plot_html(compare(*portfolios, metric='cum_excess_return').portfolio_plot()))
    sections.append("<h3>Rolling Excess Return (12m)</h3>")
    sections.append(plot_html(compare(*portfolios, metric='roll_return', months=12).dropna().portfolio_plot()))
    sections.append("<h3>Real Wealth (Total Real Cum. Compounded Return)</h3>")
    sections.append(plot_html(compare(*portfolios, metric='real_w').portfolio_plot(log=True)))

    # --- Drawdowns ---
    sections.append("<h2>Drawdowns</h2>")
    sections.append(plot_html(compare(*portfolios, metric='drawdown_series').portfolio_plot()))

    # --- Risk ---
    if single:
        sections.append("<h2>Risk</h2>")
        sections.append("<h3>Risk Contribution</h3>")
        sections.append(portfolios[0].risk_contribution().to_frame().to_html())
        sections.append("<h3>Rolling Volatility</h3>")
        sections.append(plot_html(compare(*portfolios, metric='rolling_vol').portfolio_plot()))
        sections.append("<h3>Risk Contribution</h3>")
        sections.append(plot_html(portfolios[0].rc_simple().portfolio_plot()))

        sections.append("<h2>*Extra* 12 month rolling no excess</h2>")
        sections.append(plot_html(portfolios[0].roll_return(excess=False, extras=True).portfolio_plot()))

    title = "Portfolio Comparison" if not single else portfolios[0].name
    html = f"<html><body style='font-family:sans-serif;max-width:1200px;margin:auto'><h1>{title}</h1>{''.join(sections)}</body></html>"
    if not path:
        d = Path('reports/')
        d.mkdir(exist_ok=True)
        path = d/f"{title.lower().replace(' ', '_')}_report.html"
    Path(path).write_text(html)
    print(f"Saved to {path}")